In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
# Import required libraries
import os
import kagglehub
from torch.utils.data import Dataset, DataLoader , random_split
from torchvision import transforms
import matplotlib.pyplot as plt
import os
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from PIL import Image
from glob import glob
import torch.nn as nn
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim
import pandas as pd
from torchvision import models
from torchvision.datasets import ImageFolder
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from sklearn.model_selection import train_test_split
from torchvision.datasets import ImageFolder




# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.Grayscale(3),
    transforms.ToTensor(),
      # normalize to 0-1 )
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # These random ahh values were found using ImageNet dataset, https://paperswithcode.com/dataset/imagenet
                          std=[0.229, 0.224, 0.225])

])



    # TODO: Resize to 28x28
      # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor    transforms.ToTensor(),

    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    # Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYz'
class_labels = {cls: img for img, cls in enumerate(letters)}

labels = []
for class_name, label in class_labels.items():
  labels.extend([label] * len(class_name))



# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26


print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
valid_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Check a batch of images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

labels

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get a batch of training data
data_iter = iter(train_loader)
images, labels = next(data_iter)

# CIFAR-10 class names
classes = list(train_dataset.classes)
# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = img * 0.5 + 0.1  # reverse Normalize(mean=0.5, std=0.5)
    img = np.clip(img, 0, 1)
    img = np.transpose(img.numpy(), (1, 2, 0))  # Convert (C, H, W) to (H, W, C)

    ax.imshow(img)
    ax.set_title(classes[labels[i].item()])
    ax.axis("off")

plt.show()
print("Shape of one image tensor:", images[0].shape)  # Expected: (3, 32, 32)
print("Detected classes:", classes)
print("Number of classes:", len(classes))

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

import torch
import torch.nn as nn
from torchvision import models

# Load pre-trained EfficientNet
model = models.efficientnet_v2_s(pretrained=True)  # You can choose other EfficientNet versions as well

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Modify the final fully connected layer to have 1 output (for binary classification/regression)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 26)

# Ensure the last layer is trainable
for param in model.classifier[1].parameters():
    param.requires_grad = True

# Print model to confirm changes
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(model)


# Write your code here


In [ ]:
# Write your code here


  # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)




        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)





In [ ]:
# Write your code here

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 3 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
test_losses = []

train_accuracies = []
val_accuracies = []
test_accuracies = []


# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, valid_loader, criterion, device)


    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)


    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)


    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")



In [ ]:
# Write your code here
